<a href="https://colab.research.google.com/github/haftom2012/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Chapter 1 - Introduction to Language Models</h1>
<i>Exploring the exciting field of Language AI</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)

---

This notebook is for Chapter 1 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [2]:
%%capture
!pip install transformers==4.41.2 accelerate==0.31.0

Let's break down this code:

*   `# %%capture`: This is a Jupyter/IPython magic command. When placed at the beginning of a cell, it suppresses the output of that cell. This means any text, warnings, or errors generated by the commands within the cell (like the `pip install` output) will not be displayed directly below the cell. It's often used to keep notebooks clean when the installation output is not critical.

*   `!pip install transformers==4.41.2 accelerate==0.31.0`: This line is a shell command executed from within the notebook. The `!` at the beginning indicates that the rest of the line should be run as a shell command rather than Python code.
    *   `pip install`: This is the standard command for installing Python packages.
    *   `transformers==4.41.2`: This specifies that the `transformers` library should be installed, specifically version `4.41.2`. Pinning versions ensures that the code runs with a known compatible version of the library.
    *   `accelerate==0.31.0`: Similarly, this installs the `accelerate` library, specifically version `0.31.0`.

### What is the `transformers` library?

The `transformers` library, developed by Hugging Face, is a widely used open-source library that provides thousands of pre-trained models to perform tasks on texts, images, and audio. These models are based on the transformer architecture, which revolutionized the field of natural language processing (NLP).

**Key features and uses:**

*   **Pre-trained Models:** Offers a vast collection of state-of-the-art pre-trained models (like BERT, GPT, T5, Llama, etc.) that you can use directly or fine-tune for specific tasks.
*   **Tasks:** Supports various NLP tasks such as text classification, question answering, text generation, summarization, translation, and more.
*   **Easy-to-use API:** Provides a straightforward API to load models and tokenizers, making it accessible even for beginners.
*   **Framework Agnostic:** Compatible with popular deep learning frameworks like PyTorch, TensorFlow, and JAX.
*   **Research & Production:** Used extensively in both academic research and industrial applications for building powerful AI systems.

### What is the `accelerate` library?

The `accelerate` library, also developed by Hugging Face, is a tool designed to simplify the process of running PyTorch models on various distributed training setups (like multiple GPUs, TPUs, or multi-node systems) with minimal code changes.

**Key features and uses:**

*   **Distributed Training:** Abstracts away the complexities of setting up and managing distributed training, allowing you to write your training loop as if you were running on a single device.
*   **Hardware Agnostic:** Automatically detects and configures the training environment, whether you're using a single GPU, multiple GPUs (DataParallel, DistributedDataParallel), TPUs, or mixed-precision training.
*   **Simplified API:** Provides a simple `Accelerator` object that handles device placement, gradient synchronization, and mixed-precision training automatically.
*   **Focus on Logic:** Allows developers to focus on the model and training logic rather than the boilerplate code for hardware acceleration.
*   **Integration:** Works seamlessly with the `transformers` library and other PyTorch-based projects.

In [1]:
!python --version

Python 3.11.13


In [3]:
# The current transformers version can be verified with: pip list | grep transformers
!pip list | grep transformers

sentence-transformers                 4.1.0
transformers                          4.41.2


# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [5]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Finally, we create our prompt as a user and give it to the model:

In [6]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

You are not running the flash-attention implementation, expect numerical differences.


 Why did the chicken join the band? Because it had the drumsticks!
